# ☁️ Cloud Computing Project - Incremental Streaming

This notebook demonstrates a **streaming data pipeline using PySpark** (file-based streaming, no Kafka needed).

### 📦 Data Sources:
- Web Logs
- Transactions
- Reviews
- Social Media

### 🎯 Goal: Create a **Unified Customer 360 Profile**

> ✅ **Colab-ready** — Run cells top to bottom. All fixes applied.

## Step 1: Install PySpark

In [ ]:
# Install PySpark in Colab
!pip install pyspark --quiet
print("✅ PySpark installed!")

## Step 2: Create Sample Data Folders & Files

> ⚠️ **Fix applied**: Previously failing because `data/web_logs/` folder did not exist.
> This cell creates all required folders and sample JSON files.

In [ ]:
import os
import json
from datetime import datetime, timedelta
import random

# Create all required folders
folders = ["data/web_logs", "data/transactions", "data/reviews", "data/social"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Sample customer IDs
customers = ["C001", "C002", "C003", "C004", "C005"]

def rand_time():
    dt = datetime.now() - timedelta(minutes=random.randint(0, 60))
    return dt.strftime("%Y-%m-%dT%H:%M:%S")

# --- Web Logs ---
web_events = ["click", "page_view", "search", "add_to_cart", "checkout"]
web_pages  = ["homepage", "product_page", "cart", "search_results", "checkout"]
for i, cid in enumerate(customers):
    for j in range(3):
        record = {
            "customer_id": cid,
            "event_type": random.choice(web_events),
            "value": random.choice(web_pages),
            "timestamp": rand_time()
        }
        with open(f"data/web_logs/log_{i}_{j}.json", "w") as f:
            json.dump(record, f)

# --- Transactions ---
products = ["Laptop", "Phone", "Tablet", "Headphones", "Watch"]
for i, cid in enumerate(customers):
    for j in range(2):
        record = {
            "customer_id": cid,
            "event_type": "purchase",
            "value": str(random.randint(100, 5000)),
            "timestamp": rand_time()
        }
        with open(f"data/transactions/txn_{i}_{j}.json", "w") as f:
            json.dump(record, f)

# --- Reviews ---
review_vals = ["1star", "2star", "3star", "4star", "5star"]
for i, cid in enumerate(customers):
    record = {
        "customer_id": cid,
        "event_type": "review",
        "value": random.choice(review_vals),
        "timestamp": rand_time()
    }
    with open(f"data/reviews/rev_{i}.json", "w") as f:
        json.dump(record, f)

# --- Social Media ---
social_events = ["like", "share", "comment", "post", "mention"]
for i, cid in enumerate(customers):
    for j in range(2):
        record = {
            "customer_id": cid,
            "event_type": random.choice(social_events),
            "value": f"content_{random.randint(1,100)}",
            "timestamp": rand_time()
        }
        with open(f"data/social/soc_{i}_{j}.json", "w") as f:
            json.dump(record, f)

print("✅ All data folders and sample files created!")
print(f"   web_logs:    {len(os.listdir('data/web_logs'))} files")
print(f"   transactions:{len(os.listdir('data/transactions'))} files")
print(f"   reviews:     {len(os.listdir('data/reviews'))} files")
print(f"   social:      {len(os.listdir('data/social'))} files")

## Step 3: Start SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CloudProjectStreaming") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark version: {spark.version}")
print(f"   App name: {spark.sparkContext.appName}")

## Step 4: Define Schema

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("event_type",  StringType(), True),
    StructField("value",       StringType(), True),
    StructField("timestamp",   TimestampType(), True)
])

print("✅ Schema defined:")
print(schema)

## Step 5: Load Streaming DataFrames

> ✅ Now works because data folders exist from Step 2.

In [ ]:
web_logs = spark.readStream \
    .format("json") \
    .schema(schema) \
    .load("data/web_logs/")

transactions = spark.readStream \
    .format("json") \
    .schema(schema) \
    .load("data/transactions/")

reviews = spark.readStream \
    .format("json") \
    .schema(schema) \
    .load("data/reviews/")

social = spark.readStream \
    .format("json") \
    .schema(schema) \
    .load("data/social/")

print("✅ All 4 streaming DataFrames loaded!")
print(f"   web_logs isStreaming:    {web_logs.isStreaming}")
print(f"   transactions isStreaming:{transactions.isStreaming}")
print(f"   reviews isStreaming:     {reviews.isStreaming}")
print(f"   social isStreaming:      {social.isStreaming}")

## Step 6: Aggregations Per Stream

> ⚠️ **Fix applied**: Cannot join two streaming DataFrames directly in PySpark Structured Streaming.
> Instead, each stream is written to a separate memory table, then joined as batch DataFrames.

In [ ]:
from pyspark.sql.functions import count, col

# Aggregation per stream
web_agg     = web_logs.groupBy("customer_id").agg(count("*").alias("web_activity"))
txn_agg     = transactions.groupBy("customer_id").agg(count("*").alias("transactions"))
review_agg  = reviews.groupBy("customer_id").agg(count("*").alias("reviews"))
social_agg  = social.groupBy("customer_id").agg(count("*").alias("social_activity"))

print("✅ Aggregations defined for all 4 streams!")

## Step 7: Write Each Stream to In-Memory Tables

> ✅ **Key fix**: We write each aggregated stream into a Spark in-memory table.
> Then we can join them as regular (batch) DataFrames in Step 8.
>
> `queryName` = table name used for joining later.

In [ ]:
import time

# Write each stream to in-memory tables
q1 = web_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("web_table") \
    .start()

q2 = txn_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("txn_table") \
    .start()

q3 = review_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("review_table") \
    .start()

q4 = social_agg.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("social_table") \
    .start()

print("⏳ Waiting for streams to process data...")
time.sleep(10)   # Give Spark time to process the files
print("✅ Streams running!")
print(f"   q1 (web) status:    {q1.status['message']}")
print(f"   q2 (txn) status:    {q2.status['message']}")
print(f"   q3 (review) status: {q3.status['message']}")
print(f"   q4 (social) status: {q4.status['message']}")

## Step 8: Build Unified Customer 360 Profile

> ✅ Now we read the in-memory tables as batch DataFrames and join them.
> This produces the final **Customer 360 Profile**.

In [ ]:
# Read in-memory tables as batch DataFrames
web_df    = spark.sql("SELECT * FROM web_table")
txn_df    = spark.sql("SELECT * FROM txn_table")
review_df = spark.sql("SELECT * FROM review_table")
social_df = spark.sql("SELECT * FROM social_table")

# Join all into Customer 360 Profile
customer_profile = web_df \
    .join(txn_df,    "customer_id", "outer") \
    .join(review_df, "customer_id", "outer") \
    .join(social_df, "customer_id", "outer") \
    .na.fill(0)

print("✅ Customer 360 Profile built!")
print(f"   Total customers: {customer_profile.count()}")
print()
customer_profile.show(truncate=False)

## Step 9: Save Customer 360 Profile to CSV

In [ ]:
import shutil

output_path = "output/customer_360"

# Save as CSV
customer_profile \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

# Rename the part file for easy download
import glob
part_files = glob.glob(f"{output_path}/part-*.csv")
if part_files:
    shutil.copy(part_files[0], "customer_360_profile.csv")
    print("✅ Saved to: customer_360_profile.csv")
    print("   You can download it from the Colab file browser (left panel → Files)")
else:
    print("⚠️ No output file found — check if data was processed.")

## Step 10: Stop All Streaming Queries

In [ ]:
for q in [q1, q2, q3, q4]:
    q.stop()

print("✅ All streaming queries stopped.")
print("🎉 Cloud Project Streaming Pipeline — Complete!")

## 📋 Summary of Fixes Applied

| # | Problem | Fix |
|---|---------|-----|
| 1 | `PATH_NOT_FOUND: data/web_logs/` | Step 2 creates all folders + sample JSON files |
| 2 | Cannot join two streaming DataFrames | Write each stream to in-memory table, then join as batch |
| 3 | `query.awaitTermination()` blocks execution | Replaced with `time.sleep(10)` + manual stop |
| 4 | No output saved | Added CSV export in Step 9 |

### 🏗️ Pipeline Architecture
```
JSON Files ──► Spark readStream ──► Aggregate ──► Memory Table
                                                        │
                          Customer 360 Profile ◄── SQL JOIN (batch)
                                    │
                              CSV Output File
```